In [1]:
import pandas as pd
from pathlib import Path

In [2]:
indicator_dict = {
    'PIB per capita':'NY.GDP.PCAP.CD',
    'Crecimiento del PIB':'NY.GDP.MKTP.KD.ZG',
    'Rentas del carbon % PIB':'NY.GDP.COAL.RT.ZS',
    'Empleo vulnerable':'SL.EMP.VULN.ZS',
    'Gasto P educacion':'SE.XPD.TOTL.GB.ZS',
    'var IPC':'FP.CPI.TOTL.ZG',
    'TD total':'SL.UEM.TOTL.NE.ZS',
    'TD juvenil':'SL.UEM.1524.NE.ZS', #Estimacion nacional, tambien esta de la OIT
    'TD masculina':'SL.UEM.TOTL.MA.NE.ZS',
    'TD femenina':'SL.UEM.TOTL.FE.NE.ZS',
    'TGP':'SL.TLF.CACT.NE.ZS',
    'PEA':'SL.TLF.TOTL.IN'
}
    
    

# Fetching
Only execute this group of cells once

In [4]:
import wbgapi as wb
wb.series.info(q='informality')

In [5]:
# Indicator code for GDP per capita (current US$)
dfs = {}
csv_folder = 'csv_files'
for ind in indicator_dict.keys():
    if not Path(f'{csv_folder}/{ind}.parquet').is_file():
        dfs[ind] = wb.data.DataFrame(
            indicator_dict[ind], 
            numericTimeKeys=True, 
            columns='series'
        ).reset_index()
        dfs[ind].to_parquet(f'{csv_folder}/{ind}.parquet')
    else: pass

# preprocessing

In [3]:
dfs = {string.stem: pd.read_parquet(string) for string in Path('csv_files/').glob('*.parquet')}

In [4]:
for df in dfs.keys():
    dfs[df] = dfs[df].rename(columns={k:v for v,k in indicator_dict.items()})

In [5]:
from functools import reduce
merged_df = reduce(lambda x,y: pd.merge(x,y,on=['economy','time']), dfs.values())

In [6]:
merged_df.to_excel('Macro paises.xlsx')

In [7]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17490 entries, 0 to 17489
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   economy                  17490 non-null  str    
 1   time                     17490 non-null  int64  
 2   PIB per capita           14745 non-null  float64
 3   var IPC                  11472 non-null  float64
 4   TD total                 6096 non-null   float64
 5   TD juvenil               4298 non-null   float64
 6   TD masculina             5480 non-null   float64
 7   TD femenina              5467 non-null   float64
 8   TGP                      5733 non-null   float64
 9   PEA                      8410 non-null   float64
 10  Crecimiento del PIB      14323 non-null  float64
 11  Rentas del carbon % PIB  10662 non-null  float64
 12  Empleo vulnerable        8176 non-null   float64
 13  Gasto P educacion        5739 non-null   float64
dtypes: float64(12), int64(1), str(1)


In [8]:
# Si no los tienes instalados:
# !pip install plotly ipywidgets

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

indicadores = [c for c in merged_df.columns if c not in ['economy', 'time']]
paises = sorted(merged_df['economy'].unique())

In [9]:
resumen = merged_df[indicadores].describe().T
resumen['missing_%'] = (1 - merged_df[indicadores].count() / len(merged_df)) * 100
resumen = resumen.round(2)
resumen

,count,mean,std,min,25%,50%,75%,max,missing_%
PIB per capita,14745.0,8.919930e+03,1.794588e+04,11.80,595.92,1993.59,8243.90,2.880016e+05,15.69
var IPC,11472.0,1.931000e+01,2.825900e+02,-17.64,2.37,4.78,9.52,2.377313e+04,34.41
TD total,6096.0,7.900000e+00,6.160000e+00,0.00,3.98,6.47,9.91,6.024000e+01,65.15
TD juvenil,4298.0,1.735000e+01,1.082000e+01,0.27,9.80,15.38,22.12,9.302000e+01,75.43
TD masculina,5480.0,7.460000e+00,5.780000e+00,0.00,3.81,6.11,9.23,5.634000e+01,68.67
TD femenina,5467.0,9.610000e+00,7.800000e+00,0.00,4.43,7.52,12.27,7.450000e+01,68.74
TGP,5733.0,5.990000e+01,9.880000e+00,17.99,54.34,60.59,65.43,9.669000e+01,67.22
PEA,8410.0,1.361930e+08,4.109208e+08,16668.00,1372596.25,4810507.50,33397056.75,3.736625e+09,51.92
Crecimiento del PIB,14323.0,3.680000e+00,6.180000e+00,-64.05,1.44,3.80,6.07,1.499700e+02,18.11
Rentas del carbon % PIB,10662.0,2.800000e-01,1.980000e+00,0.00,0.00,0.00,0.08,6.980000e+01,39.04


In [15]:
import plotly.io as pio
print(pio.renderers.default)

plotly_mimetype


In [16]:
import plotly.io as pio

# JupyterLab clásico
pio.renderers.default = "iframe"

# Jupyter Notebook clásico
# pio.renderers.default = "notebook"

# VSCode
# pio.renderers.default = "vscode"

# Si nada de eso funciona, usa el navegador
# pio.renderers.default = "browser"